# 直方图均衡化与直方图规定化：学生练习版

本 Notebook 基于前面 Python 基础、NumPy、Matplotlib、OpenCV 的内容，学习两种经典的数字图像增强方法：

1. 直方图均衡化（Histogram Equalization）
2. 直方图规定化（Histogram Specification / Histogram Matching）

注意：本 Notebook 是学生练习版。部分重点函数已删除具体实现，需要学生根据注释和提示补全；核心算法不得直接调用 `cv2.equalizeHist()`、`skimage.exposure.match_histograms()` 等第三方库中的现成实现。

可以使用第三方库完成以下辅助工作：

- `NumPy`：保存和处理图像数组。
- `Matplotlib`：展示图像和直方图。
- `OpenCV cv2`：读取或解码图像，但不调用其直方图均衡化函数。


练习要求：看到 `TODO` 标记的位置，就是需要学生自己补全的代码。建议先理解每个函数的输入、输出和算法步骤，再逐步运行后续测试单元。


## 1. 图像直方图基础

灰度图像中，每个像素通常是 `0` 到 `255` 之间的整数。

- `0` 表示黑色。
- `255` 表示白色。
- 中间值表示不同程度的灰色。

图像直方图统计的是每个灰度值出现的次数。如果一张图像的灰度值集中在较窄范围内，图像通常对比度较低；如果灰度值分布更分散，图像通常具有更明显的明暗层次。


## 2. 直方图均衡化原理

直方图均衡化的目标是把原图较集中的灰度分布拉伸到更宽的范围，使图像对比度增强。

基本步骤：

1. 统计原图灰度直方图。
2. 将直方图归一化，得到每个灰度值出现的概率 PDF。
3. 计算累积分布函数 CDF。
4. 根据 CDF 构造灰度映射表。
5. 用映射表把原图中的每个像素替换为新的灰度值。

对于灰度级数量为 `L=256` 的图像，常用映射公式为：

```text
s = round((L - 1) * CDF(r))
```

其中 `r` 是原始灰度值，`s` 是变换后的灰度值。


## 3. 直方图规定化原理

直方图规定化也叫直方图匹配。它的目标不是让直方图尽量均匀，而是让源图像的直方图尽可能接近一张参考图像或一个指定分布。

基本步骤：

1. 计算源图像的灰度直方图和 CDF。
2. 计算参考图像的灰度直方图和 CDF。
3. 对源图像中的每个灰度值 `r`，找到参考图像中 CDF 最接近的灰度值 `z`。
4. 建立 `r -> z` 的灰度映射表。
5. 用映射表生成规定化后的图像。

直方图均衡化可以看作一种特殊的灰度映射；直方图规定化更灵活，因为它可以让图像风格接近指定参考图像。


## 4. 导入库与读取图像

下面代码使用网络图片作为输入。如果网络无法访问，会自动生成一张低对比度灰度测试图像，保证后续算法仍然可以运行。


In [ ]:
from urllib.request import Request, urlopen

import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (6, 4)


In [ ]:
def load_image_from_url(image_url):
    """
    从网络 URL 读取一张图片，并返回 OpenCV BGR 格式图像。

    这里只使用 OpenCV 进行图像解码，不使用 OpenCV 的直方图均衡化功能。
    """
    request = Request(image_url, headers={'User-Agent': 'Mozilla/5.0'})

    with urlopen(request, timeout=15) as response:
        image_bytes = response.read()

    image_array = np.frombuffer(image_bytes, dtype=np.uint8)
    image = cv2.imdecode(image_array, cv2.IMREAD_COLOR)

    if image is None:
        raise ValueError('图片解码失败，请检查 URL 是否直接指向图片文件。')

    return image




def bgr_to_gray_by_formula(bgr_image):
    """
    手动把 BGR 彩色图像转换为灰度图像。

    常见灰度转换公式为：Gray = 0.299R + 0.587G + 0.114B。
    这里自己按照公式计算，不调用 cv2.cvtColor()。
    """
    blue = bgr_image[:, :, 0].astype(np.float32)
    green = bgr_image[:, :, 1].astype(np.float32)
    red = bgr_image[:, :, 2].astype(np.float32)

    gray = 0.299 * red + 0.587 * green + 0.114 * blue
    gray = np.clip(gray, 0, 255)

    return gray.astype(np.uint8)


In [ ]:
# 可以替换成其他网络图片 URL。
image_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/JPEG_example_flower.jpg/640px-JPEG_example_flower.jpg'

try:
    bgr_image = load_image_from_url(image_url)
    gray_image = bgr_to_gray_by_formula(bgr_image)
    print('网络图片读取成功。')
except Exception as error:
    print('网络图片读取失败，使用自动生成的低对比度测试图像。')
    print('失败原因：', error)

plt.imshow(gray_image, cmap='gray', vmin=0, vmax=255)
plt.title('Original Gray Image')
plt.axis('off')
plt.show()


## 5. 手写直方图与 CDF 函数

下面开始实现核心算法。为了帮助理解算法过程，直方图统计、CDF 计算、灰度映射都会写成独立函数。


## 5.1 学生需要补全的重点函数

本练习版删除了部分关键算法实现，需要学生补全以下函数：

1. `compute_histogram()`：手写统计灰度直方图。
2. `compute_pdf()`：根据直方图计算概率分布。
3. `compute_cdf()`：手写计算累积分布函数。
4. `create_equalization_mapping()`：根据 CDF 创建直方图均衡化映射表。
5. `apply_gray_mapping()`：使用映射表生成新图像。
6. `create_specification_mapping()`：根据源图像和参考图像的 CDF 创建规定化映射表。
7. `histogram_specification()`：组织完成直方图规定化完整流程。

补全建议：每次只完成一个函数，然后运行后面的测试单元观察错误信息和图像结果。


In [ ]:
def compute_histogram(gray, levels=256):
    """
    手写灰度直方图统计函数。需要学生补全。

    参数：
        gray: 二维灰度图像，像素值范围通常为 0 到 255。
        levels: 灰度级数量，8 位灰度图通常为 256。

    返回：
        hist: 长度为 levels 的一维数组。
              hist[i] 表示灰度值 i 在图像中出现的次数。

    实现思路：
        1. 创建一个长度为 levels 的数组 hist，并全部初始化为 0。
        2. 使用两层循环遍历图像中的每个像素。
        3. 读取当前像素的灰度值 value。
        4. 将 hist[value] 加 1。
        5. 遍历完成后返回 hist。

    注意：
        本题要求学生理解直方图统计过程，因此不要直接使用 np.histogram()。
    """
    # TODO 1: 创建长度为 levels 的直方图数组。
    # hist = ...

    # TODO 2: 获取图像高度和宽度。
    # height, width = ...

    # TODO 3: 使用双重循环遍历每个像素，并统计每个灰度值出现次数。
    # for y in range(...):
    #     for x in range(...):
    #         value = ...
    #         hist[value] += 1

    # TODO 4: 返回直方图。
    pass


def compute_pdf(hist):
    """
    根据直方图计算概率分布 PDF。需要学生补全。

    参数：
        hist: 灰度直方图，hist[i] 表示灰度值 i 出现的次数。

    返回：
        pdf: 灰度概率分布，pdf[i] 表示灰度值 i 出现的概率。

    实现思路：
        1. 计算所有像素总数 total_pixels。
        2. 如果 total_pixels 为 0，说明输入直方图无效，应抛出错误。
        3. 用 hist 除以 total_pixels，得到每个灰度值出现的概率。
    """
    # TODO 5: 计算像素总数。
    # total_pixels = ...

    # TODO 6: 判断 total_pixels 是否为 0。
    # if ...:
    #     raise ValueError(...)

    # TODO 7: 返回概率分布。
    pass


def compute_cdf(pdf):
    """
    手写累积分布函数 CDF。需要学生补全。

    参数：
        pdf: 灰度概率分布。

    返回：
        cdf: 累积分布函数。
             cdf[i] 表示灰度值小于等于 i 的像素概率总和。

    实现思路：
        1. 创建一个与 pdf 长度相同的 cdf 数组。
        2. 创建变量 running_sum，用于保存当前累计概率。
        3. 从灰度值 0 遍历到最后一个灰度值。
        4. 每次把 pdf[i] 加到 running_sum 中。
        5. 将 running_sum 保存到 cdf[i]。
        6. 返回 cdf。
    """
    # TODO 8: 创建 cdf 数组，并初始化累计和 running_sum。
    # cdf = ...
    # running_sum = ...

    # TODO 9: 使用循环计算累积概率。
    # for i in range(...):
    #     running_sum += ...
    #     cdf[i] = ...

    # TODO 10: 返回 CDF。
    pass


def plot_histogram_and_cdf(gray, title):
    """
    展示灰度图像的直方图和 CDF。

    参数：
        gray: 灰度图像。
        title: 图像标题前缀。

    说明：
        该函数依赖学生补全后的 compute_histogram、compute_pdf 和 compute_cdf。
        当这些函数正确实现后，本函数会画出直方图和累积分布曲线。
    """
    hist = compute_histogram(gray)
    pdf = compute_pdf(hist)
    cdf = compute_cdf(pdf)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.bar(range(256), hist, width=1.0)
    plt.title(title + ' Histogram')
    plt.xlabel('Gray Level')
    plt.ylabel('Count')

    plt.subplot(1, 2, 2)
    plt.plot(range(256), cdf)
    plt.title(title + ' CDF')
    plt.xlabel('Gray Level')
    plt.ylabel('Cumulative Probability')
    plt.ylim(0, 1.05)

    plt.tight_layout()
    plt.show()


### 练习提醒

从下面开始的代码单元会调用学生需要补全的函数。如果还没有完成前面的 `TODO`，运行时会出现错误或返回 `None`，这是正常现象。请先补全函数，再依次运行测试单元。


In [ ]:
plot_histogram_and_cdf(gray_image, 'Original')


## 6. 手写直方图均衡化

核心步骤：计算原图直方图、PDF、CDF，根据 CDF 创建灰度映射表，再对图像逐像素应用映射表。


In [ ]:
def create_equalization_mapping(cdf, levels=256):
    """
    根据 CDF 创建直方图均衡化映射表。需要学生补全。

    参数：
        cdf: 原图灰度累积分布函数。
        levels: 灰度级数量，8 位灰度图通常为 256。

    返回：
        mapping: 长度为 levels 的映射表。
                 mapping[r] 表示原灰度值 r 应映射到的新灰度值。

    实现思路：
        1. 创建长度为 levels 的数组 mapping。
        2. 对每个原灰度值 gray_value，取出 cdf[gray_value]。
        3. 按公式 new_value = round((levels - 1) * cdf[gray_value]) 计算新灰度值。
        4. 使用 np.clip() 确保 new_value 在 0 到 levels - 1 之间。
        5. 保存到 mapping[gray_value]。
        6. 返回 mapping。
    """
    # TODO 11: 创建映射表 mapping。
    # mapping = ...

    # TODO 12: 遍历每个灰度值，根据 CDF 计算映射后的灰度。
    # for gray_value in range(levels):
    #     new_value = ...
    #     mapping[gray_value] = ...

    # TODO 13: 返回映射表。
    pass


def apply_gray_mapping(gray, mapping):
    """
    对灰度图像应用灰度映射表。需要学生补全。

    参数：
        gray: 输入灰度图像。
        mapping: 灰度映射表。

    返回：
        result: 映射后的灰度图像。

    实现思路：
        1. 创建一个与 gray 形状相同的新图像 result。
        2. 遍历 gray 中的每一个像素。
        3. 读取旧灰度值 old_value。
        4. 使用 result[y, x] = mapping[old_value] 得到新灰度值。
        5. 返回 result。
    """
    # TODO 14: 创建结果图像 result。
    # height, width = ...
    # result = ...

    # TODO 15: 遍历每个像素，并根据 mapping 完成灰度替换。
    # for y in range(...):
    #     for x in range(...):
    #         old_value = ...
    #         result[y, x] = ...

    # TODO 16: 返回映射后的图像。
    pass


def histogram_equalization(gray, levels=256):
    """
    手写直方图均衡化函数。

    参数：
        gray: 输入灰度图像。
        levels: 灰度级数量。

    返回：
        equalized: 均衡化后的图像。
        mapping: 均衡化使用的灰度映射表。

    实现说明：
        这个函数保留完整流程，学生需要补全它调用的关键子函数。
        当 compute_histogram、compute_pdf、compute_cdf、create_equalization_mapping
        和 apply_gray_mapping 都正确后，本函数即可正常运行。
    """
    hist = compute_histogram(gray, levels)
    pdf = compute_pdf(hist)
    cdf = compute_cdf(pdf)
    mapping = create_equalization_mapping(cdf, levels)
    equalized = apply_gray_mapping(gray, mapping)

    return equalized, mapping


In [ ]:
equalized_image, equalization_mapping = histogram_equalization(gray_image)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(gray_image, cmap='gray', vmin=0, vmax=255)
plt.title('Original Gray Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(equalized_image, cmap='gray', vmin=0, vmax=255)
plt.title('Histogram Equalization')
plt.axis('off')

plt.tight_layout()
plt.show()

plot_histogram_and_cdf(equalized_image, 'Equalized')


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(256), equalization_mapping)
plt.title('Equalization Mapping')
plt.xlabel('Original Gray Level')
plt.ylabel('New Gray Level')
plt.grid(True)
plt.show()


## 7. 手写直方图规定化

直方图规定化需要一张参考图像。本练习中，参考图像不再由程序生成，而是从网络中读取一张图片，并转换为灰度图像。

源图像和参考图像的内容可以不同，尺寸也可以不同。直方图规定化关注的是灰度分布，因此只需要分别统计两张图像的灰度直方图和 CDF。


In [ ]:
# 从网络读取一张图片作为直方图规定化的参考图像。
# 可以替换成其他网络图片 URL，但应尽量使用能够直接访问的图片链接。
reference_image_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a9/Example.jpg/640px-Example.jpg'

reference_bgr_image = load_image_from_url(reference_image_url)
reference_image = bgr_to_gray_by_formula(reference_bgr_image)

plt.imshow(reference_image, cmap='gray', vmin=0, vmax=255)
plt.title('Reference Gray Image From URL')
plt.axis('off')
plt.show()

plot_histogram_and_cdf(reference_image, 'Reference')


In [ ]:
def create_specification_mapping(source_cdf, reference_cdf, levels=256):
    """
    创建直方图规定化映射表。需要学生补全。

    参数：
        source_cdf: 源图像 CDF。
        reference_cdf: 参考图像 CDF。
        levels: 灰度级数量。

    返回：
        mapping: 规定化映射表。
                 mapping[r] 表示源图像灰度 r 应映射到的参考灰度值。

    实现思路：
        1. 创建长度为 levels 的数组 mapping。
        2. 遍历源图像的每个灰度值 source_gray。
        3. 取出 source_cdf[source_gray]，记为 source_value。
        4. 在参考图像 CDF 中寻找与 source_value 最接近的位置 reference_gray。
        5. 将 reference_gray 保存到 mapping[source_gray]。
        6. 返回 mapping。

    提示：
        可以用两层循环实现。外层遍历源灰度值，内层遍历参考灰度值，
        使用 abs(source_value - reference_cdf[reference_gray]) 计算差异。
    """
    # TODO 17: 创建规定化映射表 mapping。
    # mapping = ...

    # TODO 18: 遍历源图像每个灰度值。
    # for source_gray in range(levels):
    #     source_value = ...
    #     best_reference_gray = ...
    #     smallest_difference = ...

    # TODO 19: 在参考 CDF 中找到最接近 source_value 的灰度值。
    #     for reference_gray in range(...):
    #         difference = ...
    #         if difference < smallest_difference:
    #             smallest_difference = ...
    #             best_reference_gray = ...
    #     mapping[source_gray] = ...

    # TODO 20: 返回规定化映射表。
    pass


def histogram_specification(source_gray, reference_gray, levels=256):
    """
    手写直方图规定化函数。需要学生补全。

    参数：
        source_gray: 源灰度图像。
        reference_gray: 参考灰度图像。
        levels: 灰度级数量。

    返回：
        specified: 规定化后的图像。
        mapping: 规定化使用的灰度映射表。

    实现思路：
        1. 分别计算源图像和参考图像的直方图。
        2. 分别计算源图像和参考图像的 PDF。
        3. 分别计算源图像和参考图像的 CDF。
        4. 调用 create_specification_mapping() 创建规定化映射表。
        5. 调用 apply_gray_mapping() 把映射表应用到源图像上。
        6. 返回规定化结果和映射表。
    """
    # TODO 21: 计算源图像和参考图像的直方图。
    # source_hist = ...
    # reference_hist = ...

    # TODO 22: 计算源图像和参考图像的 PDF。
    # source_pdf = ...
    # reference_pdf = ...

    # TODO 23: 计算源图像和参考图像的 CDF。
    # source_cdf = ...
    # reference_cdf = ...

    # TODO 24: 创建规定化映射表，并应用到源图像。
    # mapping = ...
    # specified = ...

    # TODO 25: 返回结果。
    pass


In [ ]:
specified_image, specification_mapping = histogram_specification(gray_image, reference_image)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(gray_image, cmap='gray', vmin=0, vmax=255)
plt.title('Original')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(reference_image, cmap='gray', vmin=0, vmax=255)
plt.title('Reference')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(specified_image, cmap='gray', vmin=0, vmax=255)
plt.title('Specification Result')
plt.axis('off')

plt.tight_layout()
plt.show()

plot_histogram_and_cdf(specified_image, 'Specified')


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(256), specification_mapping)
plt.title('Specification Mapping')
plt.xlabel('Original Gray Level')
plt.ylabel('Specified Gray Level')
plt.grid(True)
plt.show()


## 8. 对比实验：手写均衡化与 OpenCV 均衡化

OpenCV 提供了 `cv2.equalizeHist()`，可以直接完成灰度图像的直方图均衡化。

本节将前面自己写的 `histogram_equalization()` 结果与 `cv2.equalizeHist()` 的结果进行对比，包括：

1. 图像视觉效果对比。
2. 直方图对比。
3. 像素差值统计。

注意：OpenCV 常用模块中没有一个与“直方图规定化”完全对应的一行函数，因此这里主要对比直方图均衡化。直方图规定化仍然使用前面手写实现。


In [ ]:
# 使用 OpenCV 自带函数完成直方图均衡化。
# 注意：这里只用于对比实验，不属于前面算法的手写实现部分。
cv2_equalized_image = cv2.equalizeHist(gray_image)

# 计算手写结果与 OpenCV 结果之间的像素差异。
difference = np.abs(equalized_image.astype(np.int16) - cv2_equalized_image.astype(np.int16))

print('最大像素差值：', difference.max())
print('平均像素差值：', difference.mean())
print('差值不为 0 的像素数量：', np.count_nonzero(difference))
print('总像素数量：', difference.size)


In [ ]:
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.imshow(gray_image, cmap='gray', vmin=0, vmax=255)
plt.title('Original')
plt.axis('off')

plt.subplot(2, 2, 2)
plt.imshow(equalized_image, cmap='gray', vmin=0, vmax=255)
plt.title('Manual Equalization')
plt.axis('off')

plt.subplot(2, 2, 3)
plt.imshow(cv2_equalized_image, cmap='gray', vmin=0, vmax=255)
plt.title('OpenCV Equalization')
plt.axis('off')

plt.subplot(2, 2, 4)
plt.imshow(difference, cmap='gray')
plt.title('Absolute Difference')
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
manual_hist = compute_histogram(equalized_image)
cv2_hist = compute_histogram(cv2_equalized_image)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.bar(range(256), manual_hist, width=1.0)
plt.title('Manual Equalization Histogram')
plt.xlabel('Gray Level')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
plt.bar(range(256), cv2_hist, width=1.0)
plt.title('OpenCV Equalization Histogram')
plt.xlabel('Gray Level')
plt.ylabel('Count')

plt.tight_layout()
plt.show()


### 8.1 差异原因说明

如果手写结果和 OpenCV 结果不完全相同，通常是因为灰度映射公式的细节不同。

前面手写版本使用的是较直观的公式：

```text
s = round((L - 1) * CDF(r))
```

而很多工程实现会考虑非零最小 CDF，对映射范围进行进一步归一化。这样可以让最小有效灰度更接近 0，增强效果可能更明显。

因此，对比实验的重点不是要求两张结果图逐像素完全一致，而是理解：

1. 两者都基于直方图和 CDF 构造灰度映射。
2. 不同实现细节会导致少量或明显的像素差异。
3. 工程库函数通常会处理更多边界情况。


## 9. 均衡化与规定化对比

| 方法 | 目标 | 是否需要参考图像 | 典型用途 |
| --- | --- | --- | --- |
| 直方图均衡化 | 增强整体对比度，使灰度分布更分散 | 不需要 | 低对比度图像增强 |
| 直方图规定化 | 让源图像灰度分布接近指定参考分布 | 需要 | 图像风格统一、亮度分布匹配 |

注意：直方图均衡化并不总是让图像变得更自然。有时它会过度增强噪声或造成局部细节不自然。直方图规定化则取决于参考图像是否合适。


In [ ]:
plt.figure(figsize=(12, 8))

images = [gray_image, equalized_image, reference_image, specified_image]
titles = ['Original', 'Equalized', 'Reference', 'Specified']

for index, (image, title) in enumerate(zip(images, titles), start=1):
    plt.subplot(2, 2, index)
    plt.imshow(image, cmap='gray', vmin=0, vmax=255)
    plt.title(title)
    plt.axis('off')

plt.tight_layout()
plt.show()


## 10. 小结与思考题

本 Notebook 中已经手写实现了：

1. 灰度直方图统计。
2. PDF 概率分布计算。
3. CDF 累积分布函数计算。
4. 直方图均衡化映射。
5. 直方图规定化映射。

思考题：

1. 为什么直方图均衡化可以增强图像对比度？
2. 如果原图本身对比度已经很高，继续做直方图均衡化可能会发生什么？
3. 直方图规定化为什么需要参考图像？
4. 如果参考图像的亮度分布很极端，规定化结果会怎样？
5. 本 Notebook 只处理灰度图像。如果要处理彩色图像，可以考虑对哪些通道进行处理？
